# Role-based matched trees (human-readable)

This notebook:
- reads role definitions from `config/userconfig.json`
- loads feature-importance rows from one epsilon folder
- picks the best-matching tree for each role
- prints readable explanations and example decision paths

The repository contains eps0.005 and eps_0.010(EPSILON used as default here). For larget thresholds, run `generateSet.py`.


In [1]:
import os
import json
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')
# sklearn tree internals use -2 for undefined/sentinel nodes.
TREE_UNDEFINED = -2


def find_project_root(start_dir: Path) -> Path:
    for p in [start_dir] + list(start_dir.parents):
        if (p / "config" / "userconfig.json").exists() and (p / "data").exists():
            return p
    return start_dir


def load_json_flexible(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Config file is empty: {path}")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Handles cases like trailing commas or single-quoted content.
        return ast.literal_eval(text)


PROJECT_ROOT = find_project_root(Path.cwd())
USER_CFG_PATH = PROJECT_ROOT / "config" / "userconfig.json"





In [2]:
### CONFIG
role_features = load_json_flexible(USER_CFG_PATH)

# Same root as generateSet.py outputs
OUTPUT_ROOT = PROJECT_ROOT / "data" / "rashomonSet"
EPSILON_TO_USE = "0.010"  # change if needed, e.g. "0.050"
EPS_DIR = OUTPUT_ROOT / f"eps_{EPSILON_TO_USE}"

if not os.path.exists(EPS_DIR):
    raise FileNotFoundError(f"Epsilon folder not found: {EPS_DIR}")

FI_PATH = os.path.join(EPS_DIR, "feature_importances.csv")
if not os.path.exists(FI_PATH):
    raise FileNotFoundError(
        f"feature_importances.csv not found in {EPS_DIR}. "
        "Run analyseVariability.py first."
    )

fi = pd.read_csv(FI_PATH)


df = pd.read_csv("data/framingham_preproc.csv", index_col=0)
y = df["TenYearCHD"]
X = df.drop(columns=["TenYearCHD"]).fillna(df.drop(columns=["TenYearCHD"]).median())
feature_names = X.columns.tolist()

_, X_test, _, _ = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=10
)


def score_tree(row, features):
    present = [f for f in features if f in row.index]
    if not present:
        return 0.0
    return float(row[present].sum())


def extract_readable_rules(tree, x_values):
    tree_ = tree.tree_
    feature = tree_.feature
    threshold = tree_.threshold

    node_indicator = tree.decision_path(x_values)
    node_index = node_indicator.indices[
        node_indicator.indptr[0]: node_indicator.indptr[1]
    ]

    readable = []
    for node_id in node_index:
        if feature[node_id] == TREE_UNDEFINED:
            continue

        feat_idx = feature[node_id]
        feat_name = feature_names[feat_idx]
        thr = threshold[node_id]
        value = x_values[0, feat_idx]

        if value <= thr:
            readable.append(f"{feat_name} is at most {thr:.2f}")
        else:
            readable.append(f"{feat_name} is above {thr:.2f}")

    return readable


def risk_band(prob):
    if prob < 0.20:
        return "low"
    return "high"


In [3]:
# User-wise matching and explanation

for role, feats in role_features.items():
    print("\n" + "=" * 68)
    print(f"User: {role}")
    print("=" * 68)
    print("User aligned features:", ", ".join(feats))

    role_scores = fi.apply(lambda row: score_tree(row, feats), axis=1)
    best_tree_idx = int(role_scores.idxmax())
    best_score = float(role_scores.iloc[best_tree_idx])

    tree_path = os.path.join(EPS_DIR, f"tree_{best_tree_idx}.pkl")
    with open(tree_path, "rb") as f:
        tree = pickle.load(f)

    print(f"Matched tree: tree_{best_tree_idx}.pkl")
    print(f"Match score (sum of user-feature importances): {best_score:.4f}")

    row = fi.loc[best_tree_idx]
    ranked = row[feature_names].sort_values(ascending=False)

    print("\nTop 5 important factors in this tree:")
    for feat, val in ranked.head(5).items():
        print(f"- {feat}: {val:.4f}")

    print("\nUser features in this tree:")
    for feat in feats:
        if feat in ranked.index:
            print(f"- {feat}: {ranked[feat]:.4f}")
        else:
            print(f"- {feat}: not present in model features")

    print("\nExample decisions (first 3 test people):")
    for i in range(1):
        x = X_test.iloc[i:i+1].values
        prob = float(tree.predict_proba(x)[0, 1])
        band = risk_band(prob)
        rules = extract_readable_rules(tree, x)

        print(f"\nPerson {i+1}: predicted risk = {prob:.3f} ({band})")
        print("Decision path:")
        for rule in rules:
            print(f"  - {rule}")



User: doctor
User aligned features: MAP, totChol, diabetes_grade
Matched tree: tree_19.pkl
Match score (sum of user-feature importances): 0.3378

Top 5 important factors in this tree:
- age: 0.2993
- MAP: 0.2577
- male: 0.0962
- prevalentHyp: 0.0945
- totChol: 0.0801

User features in this tree:
- MAP: 0.2577
- totChol: 0.0801
- diabetes_grade: 0.0000

Example decisions (first 3 test people):

Person 1: predicted risk = 0.129 (low)
Decision path:
  - age is above 48.50
  - prevalentHyp is above 0.50
  - MAP is at most 128.25
  - male is at most 0.50
  - age is at most 54.50

User: patient
User aligned features: cigsPerDay, BMI
Matched tree: tree_39.pkl
Match score (sum of user-feature importances): 0.1310

Top 5 important factors in this tree:
- age: 0.3000
- MAP: 0.2069
- male: 0.1148
- prevalentHyp: 0.0909
- totChol: 0.0771

User features in this tree:
- cigsPerDay: 0.0574
- BMI: 0.0735

Example decisions (first 3 test people):

Person 1: predicted risk = 0.129 (low)
Decision path:
